# Notebook 15 — Step 2: Multi-trial Benchmarking and Statistical Tests

This notebook runs the multi-trial benchmarks required by Step 2 (
n=5 trials per prompt), computes summary statistics, and performs significance tests
(McNemar for paired success/failure; paired t-test for continuous metrics).

NOTE: Cells are *not* executed here — run them locally. Ensure you have `scipy` installed.

## Requirements & Notes
- Uses `AblationStudy` from `src/evaluation/ablation.py`.
- Default configuration in `config/config.json` may point to local Ollama.
- To reproduce on AWS Bedrock, update `config/config.json` to use AWS provider backups.
- This notebook runs the full study with `num_trials=5` (adjustable).

In [1]:
# Setup imports and helper libs
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from pprint import pprint

# Resolve project root robustly
current_dir = Path('.').resolve()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
elif (current_dir / 'notebooks').exists():
    project_root = current_dir
else:
    project_root = current_dir
    for parent in [current_dir] + list(current_dir.parents):
        if parent.name == 'Cirq-RAG-Code-Assistant':
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Stats imports (scipy required for t-test and chi2)
try:
    from scipy import stats
    from scipy.stats import chi2
except Exception as e:
    print('Scipy not available. Install via pip install scipy before running tests.')
    stats = None
    chi2 = None

# Project imports (AblationStudy + utilities)
from src.evaluation.ablation import AblationStudy, VARIANT_LABELS
from src.evaluation.benchmark import load_benchmark_prompts
from src.evaluation.metrics import compute_statistics, wilson_ci

# Paths
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)
print(f'Working directory: {project_root}')
print('Notebook prepared. Run cells to execute the study.')

c:\Study Material\FYP\QCanvas-Project\QCanvas\qasm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant
Notebook prepared. Run cells to execute the study.


In [2]:
# Load benchmark prompts (code tiers only)
benchmark_cases = load_benchmark_prompts(exclude_explanation=True)
print(f'Loaded {len(benchmark_cases)} benchmark prompts (code tiers)')

# Quick preview
for p in benchmark_cases[:3]:
    pprint({ 'id': p.get('id'), 'tier': p.get('tier'), 'query': p.get('query') })


2026-06-02 14:37:54.145 | INFO     | config.config_loader:load:118 - ✅ Loaded configuration from C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\config\config.json
2026-06-02 14:37:54.149 | DEBUG    | config.config_loader:create_directories:304 - Created all necessary directories
2026-06-02 14:37:54.152 | INFO     | src.evaluation.benchmark:load_benchmark_prompts:87 - Loaded 25 benchmark prompts from data\datasets\benchmark_prompts_v2.jsonl


Loaded 20 benchmark prompts (code tiers)
{'id': 'BM-001',
 'query': 'Create a 2-qubit Bell state circuit with measurement',
 'tier': 'basic'}
{'id': 'BM-002',
 'query': 'Create a 3-qubit GHZ state circuit with measurement',
 'tier': 'basic'}
{'id': 'BM-003',
 'query': 'Apply X then H on a single qubit and measure',
 'tier': 'basic'}


In [3]:
# Configuration for the run (edit before executing)
NUM_TRIALS = 5            # Step 2 requires n=5 trials per prompt
MAX_BENCHMARK_CASES = None  # None = all prompts (25); set to int for quicker dev runs
VARIANTS = list(VARIANT_LABELS.keys())  # Default variants defined in ablation module
SAVE_PATH = results_dir / 'ablation_results_step2.json'

# Instantiate the study (no run yet)
study = AblationStudy(benchmark_cases=benchmark_cases)
print('AblationStudy initialized with', len(study.benchmark_cases), 'cases')

# To run: uncomment the next lines and execute this cell. (Notebook will run the full study.)
# print(f'Running study: variants={VARIANTS} | cases={MAX_BENCHMARK_CASES or len(benchmark_cases)} | trials={NUM_TRIALS}')
# results = study.run_study(variants=VARIANTS, max_cases=MAX_BENCHMARK_CASES, num_trials=NUM_TRIALS)
# study.save_results(SAVE_PATH, results)
# print('Saved results to', SAVE_PATH)


2026-06-02 14:37:54.166 | INFO     | src.rag.embeddings:_init_local:134 - Loading embedding model: BAAI/bge-base-en-v1.5
2026-06-02 14:37:54.166 | INFO     | src.rag.embeddings:_init_local:135 - Using device: cpu
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4190.47it/s]
2026-06-02 14:38:03.501 | INFO     | src.rag.embeddings:_init_local:142 - ✅ Embedding model loaded successfully
C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\src\rag\embeddings.py:146: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()
2026-06-02 14:38:03.509 | INFO     | src.rag.embeddings:_init_local:147 - Embedding dimension: 768
2026-06-02 14:38:03.509 | INFO     | src.rag.vector_store:_init_faiss:155 - Initialized FAISS index
2026-06-02 14:38:03.509 | INFO     | src.rag.vector_store:__init__:136 - Initialized VectorStore with faiss backend
2026-06-02 14:38:

AblationStudy initialized with 20 cases


In [4]:
# Aggregate and prepare modes dict for visualization / analysis
# After running the study, set `results` variable to returned dict or load from SAVE_PATH

# Example: load previously saved results if present
if SAVE_PATH.exists():
    results = json.loads(SAVE_PATH.read_text(encoding='utf-8'))
    study.results = results
    print('Loaded results from', SAVE_PATH)
else:
    print('No results file found at', SAVE_PATH, 
)

# Aggregate into modes dict used by notebook visualizations
modes = study.aggregate_to_modes_dict(study.results if hasattr(study, 'results') else results)
mode_names = list(modes.keys())
print('Available modes:', mode_names)


Loaded results from C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\results\ablation_results_step2.json
Available modes: ['Ideal System (Hypothetical)', 'Full System', 'No RAG', 'No Validator', 'No Optimizer', 'No Final Validator', 'Only Designer']


In [5]:
# Significance tests helpers: McNemar (paired binary) and paired t-test (continuous)
from math import isclose

def mcnemar_test_from_pairlists(a_success: list, b_success: list):
    """Compute McNemar test comparing two paired binary outcomes.
    Returns dict with b, c, statistic, p_value."""
    assert len(a_success) == len(b_success), 'Length mismatch'
    b = 0  # a=1,b=0
    c = 0  # a=0,b=1
    for a, bval in zip(a_success, b_success):
        if a and not bval:
            b += 1
        elif not a and bval:
            c += 1
    n = b + c
    if n == 0:
        return {'b': b, 'c': c, 'statistic': 0.0, 'p_value': 1.0}
    # Continuity-corrected McNemar chi-square statistic
    stat = (abs(b - c) - 1)**2 / n if n > 0 else 0.0
    if chi2 is None:
        p = None
    else:
        p = 1 - chi2.cdf(stat, df=1)
    return {'b': b, 'c': c, 'statistic': stat, 'p_value': p}

def paired_ttest(list_a: list, list_b: list):
    """Paired t-test for continuous paired samples. Returns t-stat and p-value."""
    if stats is None:
        return {'t_stat': None, 'p_value': None}
    t_stat, p = stats.ttest_rel(list_a, list_b, nan_policy='omit')
    return {'t_stat': float(t_stat) if t_stat is not None else None, 'p_value': float(p) if p is not None else None}

print('Significance helpers defined: mcnemar_test_from_pairlists(), paired_ttest()')

# Example usage (uncomment to run after `results` exists):
# Compare 'full' vs 'no_rag' successes across all trials (flattened per-case trials)
# a_details = [d for d in results['full']['details']]
# b_details = [d for d in results['no_rag']['details']]
# a_success = [1 if d['success'] else 0 for d in a_details]
# b_success = [1 if d['success'] else 0 for d in b_details]
# print(mcnemar_test_from_pairlists(a_success, b_success))
# For latency paired t-test: paired_ttest([d['latency'] for d in a_details], [d['latency'] for d in b_details])


Significance helpers defined: mcnemar_test_from_pairlists(), paired_ttest()


In [6]:
# Produce a concise summary CSV for paper tables (after results exist)
if not (SAVE_PATH.exists() or hasattr(study, 'results')):
    print('No results available yet. Run the study cell first or place results at', SAVE_PATH)
else:
    res = study.results if (hasattr(study, 'results') and study.results) else json.loads(SAVE_PATH.read_text(encoding='utf-8'))
    rows = []
    for variant, data in res.items():
        rows.append({
            'variant': variant,
            'label': data.get('label') or VARIANT_LABELS.get(variant, variant),
            'total_cases': data.get('total_cases'),
            'success_rate': data.get('success_rate'),
            'success_ci_low': data.get('success_rate_ci', {}).get('ci_lower'),
            'success_ci_high': data.get('success_rate_ci', {}).get('ci_upper'),
            'validation_rate': data.get('validation_rate'),
            'avg_latency': data.get('avg_latency'),
            'latency_std': data.get('latency_stats', {}).get('std'),
            'avg_code_quality': data.get('code_quality_stats', {}).get('mean'),
            'avg_circuit_depth': data.get('circuit_depth_stats', {}).get('mean'),
            'circuit_depth_std': data.get('circuit_depth_stats', {}).get('std'),
            'avg_gate_count': data.get('gate_count_stats', {}).get('mean'),
            'gate_count_std': data.get('gate_count_stats', {}).get('std'),
            'avg_two_qubit_gates': data.get('two_qubit_gate_stats', {}).get('mean'),
        })
    df_summary = pd.DataFrame(rows).sort_values('label')
    out_csv = results_dir / 'ablation_summary_step2.csv'
    df_summary.to_csv(out_csv, index=False)
    print('Saved summary CSV to', out_csv)
    display(df_summary)


Saved summary CSV to C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\results\ablation_summary_step2.csv


,variant,label,total_cases,success_rate,success_ci_low,success_ci_high,validation_rate,avg_latency,latency_std,avg_code_quality,avg_circuit_depth,circuit_depth_std,avg_gate_count,gate_count_std,avg_two_qubit_gates
1,full,Full System,20,0.82,0.733325,0.882999,0.82,8.539083,1.357996,0.902199,11.329268,2.464839,22.207317,4.938480,3.390244
0,ideal,Ideal System (Hypothetical),20,0.96,0.901628,0.984337,0.96,1.494899,0.271741,0.969020,4.552083,0.938866,9.614583,1.871503,1.447917
5,no_final_validator,No Final Validator,20,0.78,0.689295,0.849988,0.78,7.554096,1.048516,0.860182,11.141026,2.254717,20.628205,4.789352,3.435897
4,no_optimizer,No Optimizer,20,0.86,0.778626,0.914737,0.82,6.517501,0.986483,0.905487,25.024390,4.455474,42.634146,8.371781,11.609756
2,no_rag,No RAG,20,0.55,0.452444,0.643856,0.53,7.227819,1.156794,0.726059,14.452830,2.763540,27.716981,7.409395,5.301887
3,no_validator,No Validator,20,0.77,0.678454,0.841568,0.75,7.006955,1.129791,0.801235,12.720000,2.496267,22.786667,4.630374,4.373333
6,minimal,Only Designer,20,0.59,0.492013,0.681328,0.58,5.083025,0.696306,0.634443,15.465517,3.062023,30.275862,4.990493,6.706897


## Provider note
The repository configuration may point to local Ollama models (for offline experiments).
If you prefer to run on AWS Bedrock for paper reproduction, update `config/config.json` to set the agent model providers to `aws` and use the relevant model names, then re-run the study cells.

# 15. Step 2 — Multi-trial Benchmark Analysis (n=5)

This notebook runs (or loads) the multi-trial ablation experiments required by Step 2 of the revision plan, computes per-prompt statistics (mean, std, 95% CI), and performs statistical tests: McNemar's test for paired binary outcomes and paired t-tests for continuous metrics.

Notes:
- Default is NOT to run the full ablation (set `RUN_ABLATION = True` in the cell below).
- Uses the same `AblationStudy` implementation as Notebook 12 but enforces `num_trials=5` and adds significance tests.
- Designed to be run locally (Ollama) or against AWS Bedrock depending on your `config/config.json` settings.

In [7]:
# Setup & imports
import sys
import os
from pathlib import Path
import json
import math
import time
import pandas as pd
import numpy as np

# Resolve project root robustly
current_dir = Path('.').resolve()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
elif (current_dir / 'notebooks').exists():
    project_root = current_dir
else:
    project_root = current_dir
    for parent in [current_dir] + list(current_dir.parents):
        if parent.name == 'Cirq-RAG-Code-Assistant':
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

from src.evaluation.ablation import AblationStudy, VARIANT_LABELS
from src.evaluation.benchmark import load_benchmark_prompts

# Results path for this step
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)
ABLATION_CACHE = results_dir / 'ablation_results_step2.json'

print('Setup complete')

Working directory: C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant
Setup complete


In [8]:
# Statistical test helpers (McNemar + paired t-test with sciPy fallback)
try:
    from scipy import stats
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def mcnemar_test(b: int, c: int, continuity: bool = True) -> dict:
    """Compute McNemar's test statistic and approximate p-value.
    b: cases full=1 variant=0 (full success, variant fail)
    c: cases full=0 variant=1 (full fail, variant success)
    Returns dict {statistic, p_value, b, c}
    Uses continuity correction when True.
    """
    n = b + c
    if n == 0:
        return {'statistic': 0.0, 'p_value': 1.0, 'b': b, 'c': c}
    if _HAS_SCIPY:
        # Use scipy exact/midp if available (use asymptotic with continuity correction here)
        stat = ((abs(b - c) - (1.0 if continuity else 0.0)) ** 2) / n
        p = 1.0 - stats.chi2.cdf(stat, df=1)
        return {'statistic': stat, 'p_value': p, 'b': b, 'c': c}
    else:
        stat = ((abs(b - c) - (1.0 if continuity else 0.0)) ** 2) / n
        # approximate p-value from chi-square(1) using survival function with math only (approx)
        # use the chi-square cdf approximation via math.erf is not trivial; fallback to normal approximation for large n
        try:
            # Wilson-Hilferty transform for chi-square to normal approx (not ideal for small n)
            # compute p using survival via exp(-stat/2) approximation
            p = math.exp(-stat / 2.0)
            p = min(1.0, max(0.0, p))
        except Exception:
            p = 1.0
        return {'statistic': stat, 'p_value': p, 'b': b, 'c': c}

def paired_t_test(x: list, y: list) -> dict:
    """Paired t-test between two lists of equal length. Returns t-stat, df, two-sided p.
    Falls back to normal approximation if SciPy not available.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape or x.size == 0:
        return {'t_stat': 0.0, 'p_value': 1.0, 'df': 0}
    d = x - y
    n = d.size
    mean_d = float(d.mean())
    if n == 1:
        return {'t_stat': 0.0, 'p_value': 1.0, 'df': 0}
    sd_d = float(d.std(ddof=1))
    se = sd_d / math.sqrt(n)
    if se == 0.0:
        return {'t_stat': 0.0, 'p_value': 1.0, 'df': n-1}
    t_stat = mean_d / se
    df = n - 1
    if _HAS_SCIPY:
        p = stats.t.sf(abs(t_stat), df) * 2.0
    else:
        # approximate p using normal distribution tail (acceptable for moderate n)
        p = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(t_stat) / math.sqrt(2.0))))
    return {'t_stat': float(t_stat), 'p_value': float(p), 'df': int(df)}

In [9]:
# Run or load ablation study results (no execution by default)
RUN_ABLATION = False  # Set True to run the full ablation (prepare for many API calls)
NUM_TRIALS = 5
MAX_BENCHMARK_CASES = None  # None = all 25 prompts

if RUN_ABLATION:
    print(f'Running ablation with num_trials={NUM_TRIALS}, max_cases={MAX_BENCHMARK_CASES}')
    prompts = load_benchmark_prompts(exclude_explanation=True)
    study = AblationStudy(benchmark_cases=prompts)
    variants = list(VARIANT_LABELS.keys())
    results = study.run_study(variants=variants, max_cases=MAX_BENCHMARK_CASES, num_trials=NUM_TRIALS)
    study.save_results(ABLATION_CACHE, results)
    print(f'Saved ablation results to {ABLATION_CACHE}')
else:
    if ABLATION_CACHE.exists():
        print(f'Loading cached ablation results from {ABLATION_CACHE}')
        with open(ABLATION_CACHE, encoding='utf-8') as f:
            results = json.load(f)
    else:
        raise FileNotFoundError(f'No cache at {ABLATION_CACHE}; set RUN_ABLATION=True to create it')

Loading cached ablation results from C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\results\ablation_results_step2.json


In [10]:
# Helper: build paired arrays between full system and another variant
def build_paired_outcomes(results: dict, full_key: str = 'full', compare_key: str = 'no_rag') -> dict:
    """Return paired lists for binary and continuous metrics between full and compare variant.
    Expects results[variant]['details'] to contain per-trial entries in same order.
    Returns dict with paired success, latency, quality, depth, and gate counts.
    """
    if full_key not in results or compare_key not in results:
        raise KeyError('Variant keys not found in results')

    full_details = results[full_key]['details']
    comp_details = results[compare_key]['details']

    # Build mapping by (id, trial index) — the study stored details in sequential order: cases * trials
    def index_map(details):
        m = {}
        for idx, d in enumerate(details):
            key = (d.get('id'), idx)
            m[key] = d
        return m

    f_map = index_map(full_details)
    c_map = index_map(comp_details)

    paired_success = []
    paired_latency = []
    paired_quality = []
    paired_depth = []
    paired_gates = []

    # iterate over intersection of keys to ensure strict pairing
    keys = sorted(set(f_map.keys()) & set(c_map.keys()))
    for k in keys:
        f = f_map[k]
        c = c_map[k]
        f_s = 1 if f.get('success') else 0
        c_s = 1 if c.get('success') else 0
        paired_success.append((f_s, c_s))
        paired_latency.append((float(f.get('latency') or 0.0), float(c.get('latency') or 0.0)))
        paired_quality.append((float(f.get('code_quality_score') or 0.0), float(c.get('code_quality_score') or 0.0)))
        paired_depth.append((float(f.get('circuit_depth') or 0.0), float(c.get('circuit_depth') or 0.0)))
        paired_gates.append((float(f.get('num_gates') or 0.0), float(c.get('num_gates') or 0.0)))

    return {
        'paired_success': paired_success,
        'paired_latency': paired_latency,
        'paired_quality': paired_quality,
        'paired_depth': paired_depth,
        'paired_gates': paired_gates
    }


In [11]:
# Perform statistical comparisons: Full System vs each other variant
results_keys = list(results.keys())
if 'full' not in results_keys:
    print('Warning: "full" variant not found; available variants:', results_keys)

comparisons = []
for variant_key in results_keys:
    if variant_key == 'full':
        continue
    try:
        paired = build_paired_outcomes(results, full_key='full', compare_key=variant_key)
    except KeyError:
        print(f'Skipping comparison; missing variant {variant_key}')
        continue

    # McNemar counts
    b = sum(1 for f, c in paired['paired_success'] if f == 1 and c == 0)
    c = sum(1 for f, c in paired['paired_success'] if f == 0 and c == 1)
    m_res = mcnemar_test(b, c, continuity=True)

    # Paired t-test on latency (full, comp)
    latency_full = [f for f, _ in paired['paired_latency']]
    latency_comp = [c for _, c in paired['paired_latency']]
    t_latency = paired_t_test(latency_full, latency_comp)

    # Paired t-test on code quality
    qual_full = [f for f, _ in paired['paired_quality']]
    qual_comp = [c for _, c in paired['paired_quality']]
    t_quality = paired_t_test(qual_full, qual_comp)

    # Paired t-test on circuit depth
    depth_full = [f for f, _ in paired['paired_depth']]
    depth_comp = [c for _, c in paired['paired_depth']]
    t_depth = paired_t_test(depth_full, depth_comp)

    # Paired t-test on gate count
    gates_full = [f for f, _ in paired['paired_gates']]
    gates_comp = [c for _, c in paired['paired_gates']]
    t_gates = paired_t_test(gates_full, gates_comp)

    comparisons.append({
        'variant': variant_key,
        'label': results[variant_key].get('label', variant_key),
        'n_pairs': len(paired['paired_success']),
        'mcnemar_b': b,
        'mcnemar_c': c,
        'mcnemar_stat': m_res['statistic'],
        'mcnemar_p': m_res['p_value'],
        'latency_t': t_latency['t_stat'],
        'latency_df': t_latency['df'],
        'latency_p': t_latency['p_value'],
        'quality_t': t_quality['t_stat'],
        'quality_df': t_quality['df'],
        'quality_p': t_quality['p_value'],
        'depth_t': t_depth['t_stat'],
        'depth_df': t_depth['df'],
        'depth_p': t_depth['p_value'],
        'gates_t': t_gates['t_stat'],
        'gates_df': t_gates['df'],
        'gates_p': t_gates['p_value'],
    })

# Summarize into DataFrame and save
df_comp = pd.DataFrame(comparisons)
out_path = results_dir / 'step2_statistical_comparisons.csv'
df_comp.to_csv(out_path, index=False)
print(f'Statistical comparisons saved to: {out_path}')
df_comp

Statistical comparisons saved to: C:\Study Material\FYP\QCanvas-Project\QCanvas\Cirq-RAG-Code-Assistant\results\step2_statistical_comparisons.csv


,variant,label,n_pairs,mcnemar_b,mcnemar_c,mcnemar_stat,mcnemar_p,latency_t,latency_df,latency_p,quality_t,quality_df,quality_p,depth_t,depth_df,depth_p,gates_t,gates_df,gates_p
0,ideal,Ideal System (Hypothetical),100,3,17,8.450000,0.003650,51.043998,99,6.407347e-73,-4.148248,99,7.088681e-05,9.733687,99,4.169156e-16,9.000954,99,1.648881e-14
1,no_rag,No RAG,100,38,11,13.795918,0.000204,7.113150,99,1.801572e-10,8.320624,99,4.911818e-13,1.763555,99,8.089206e-02,1.917985,99,5.799458e-02
2,no_validator,No Validator,100,16,11,0.592593,0.441418,8.450429,99,2.576844e-13,5.818426,99,7.328333e-08,-0.335999,99,7.375829e-01,0.798755,99,4.263449e-01
3,no_optimizer,No Optimizer,100,11,15,0.346154,0.556298,13.017841,99,3.585249e-23,-0.174771,99,8.616169e-01,-10.002992,99,1.077660e-16,-7.816532,99,5.919817e-12
4,no_final_validator,No Final Validator,100,14,10,0.375000,0.540291,5.575630,99,2.139618e-07,2.275803,99,2.501247e-02,0.955463,99,3.416708e-01,1.684829,99,9.517164e-02
5,minimal,Only Designer,100,34,11,10.755556,0.001040,26.680119,99,5.280265e-47,16.686176,99,1.578330e-30,0.345842,99,7.301950e-01,0.355295,99,7.231244e-01


## How to run

1. If you want to execute the full multi-trial ablation (25 prompts × 5 trials × variants), set `RUN_ABLATION = True` in the appropriate cell, then run cells in order. Expect many LLM calls and runtime — ensure Ollama or your AWS credentials are available.

2. If you already ran Notebook 12 and have a cached `results/ablation_results_step2.json`, leave `RUN_ABLATION = False` and the notebook will load it and compute tests.

3. Review `step2_statistical_comparisons.csv` for McNemar p-values (binary success differences) and paired t-test p-values for latency and quality.

Notes on interpretation:
- McNemar p < 0.05 suggests a significant change in binary success between Full and the variant.
- Paired t-test p < 0.05 indicates a significant difference in continuous metric (latency or code quality).
- If SciPy is not installed, p-values are approximated. Install `scipy` for exact tests: `pip install scipy`.